In [1]:
#!hdfs dfs -mkdir -p /data

In [2]:
###
!hdfs dfs -put C:\Users\vongo\Final_BigData\flights.csv /data/
!hdfs dfs -put C:\Users\vongo\Final_BigData\airlines.csv /data/
!hdfs dfs -put C:\Users\vongo\Final_BigData\airports.csv /data/
###

put: `/data/flights.csv': File exists
put: `/data/airlines.csv': File exists
put: `/data/airports.csv': File exists


In [3]:
!hdfs dfs -ls /data/

Found 3 items
-rw-r--r--   1 vongo supergroup        359 2026-06-12 00:45 /data/airlines.csv
-rw-r--r--   1 vongo supergroup      23867 2026-06-12 00:45 /data/airports.csv
-rw-r--r--   1 vongo supergroup  592406591 2026-06-12 00:45 /data/flights.csv


In [1]:
import os
import shutil

print("JAVA_HOME =", os.environ.get("JAVA_HOME"))
print("java =", shutil.which("java"))
print("spark-submit =", shutil.which("spark-submit"))

JAVA_HOME = C:\java\openjdk-17.0.18b8
java = C:\java\openjdk-17.0.18b8\bin\java.EXE
spark-submit = C:\Users\vongo\Final_BigData\.venv\Scripts\spark-submit.CMD


In [2]:
import pyspark
print("PySpark:", pyspark.__version__)

PySpark: 4.1.1


In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .appName("Flight Data Analysis") \
    .getOrCreate()

In [3]:
flights_df  = spark.read.csv("hdfs://localhost:9000/data/flights.csv",   header=True, inferSchema=True)
airlines_df = spark.read.csv("hdfs://localhost:9000/data/airlines.csv",  header=True, inferSchema=True)
airports_df = spark.read.csv("hdfs://localhost:9000/data/airports.csv",  header=True, inferSchema=True)

flights_df.createOrReplaceTempView("flights")
airlines_df.createOrReplaceTempView("airlines")
airports_df.createOrReplaceTempView("airports")

In [6]:
import pandas as pd

# Q1 - Which airline has the most delays? A month-by-month comparison

In [9]:
query1 = spark.sql("""
    WITH monthly_stats AS (
        SELECT
            al.AIRLINE AS AirlineName,
            f.MONTH,
            COUNT(*) AS total_flights,
            ROUND(AVG(f.DEPARTURE_DELAY), 2) AS avg_dep_delay,
            ROUND(AVG(f.ARRIVAL_DELAY),   2) AS avg_arr_delay,
            SUM(CASE WHEN f.ARRIVAL_DELAY > 15 THEN 1 ELSE 0 END) AS delayed_flights_count,
            ROUND(
                SUM(CASE WHEN f.ARRIVAL_DELAY > 15 THEN 1 ELSE 0 END)
                * 100 / COUNT(*), 1) AS pct_delayed
        FROM flights f
        JOIN airlines al ON f.AIRLINE = al.IATA_CODE
        WHERE f.CANCELLED = 0
          AND f.DEPARTURE_DELAY IS NOT NULL
          AND f.ARRIVAL_DELAY   IS NOT NULL
        GROUP BY al.AIRLINE, f.MONTH
        HAVING COUNT(*) >= 200
    ),
    ranked_stats AS (
        SELECT
            AirlineName,
            RANK() OVER (PARTITION BY MONTH ORDER BY avg_arr_delay DESC) AS rank_worst,
            MONTH,
            total_flights,
            avg_dep_delay,
            avg_arr_delay,
            delayed_flights_count,
            pct_delayed
        FROM monthly_stats
    )
    SELECT
        AirlineName,
        MONTH,
        total_flights,
        avg_dep_delay,
        avg_arr_delay,
        delayed_flights_count,
        pct_delayed,
        rank_worst
    FROM ranked_stats
    WHERE rank_worst = 1
    ORDER BY MONTH
""")
print("\n Q1 - Which airline has the most delays? A month-by-month comparison ")
df1 = query1.toPandas()
df1

,AirlineName,MONTH,total_flights,avg_dep_delay,avg_arr_delay,delayed_flights_count,pct_delayed,rank_worst
0,Frontier Airlines Inc.,1,6735,17.91,18.36,2091,31.0,1
1,Frontier Airlines Inc.,2,5691,25.62,27.42,2215,38.9,1
2,Frontier Airlines Inc.,3,6880,19.87,20.07,2280,33.1,1
3,Frontier Airlines Inc.,4,7092,11.56,12.64,1845,26.0,1
4,Spirit Air Lines,5,9800,23.52,22.40,3550,36.2,1
5,Spirit Air Lines,6,9325,36.06,35.56,4338,46.5,1
6,Frontier Airlines Inc.,7,8038,13.85,14.46,2182,27.1,1
7,Spirit Air Lines,8,10227,22.18,20.52,3503,34.3,1
8,Spirit Air Lines,9,9868,9.66,8.00,2147,21.8,1
9,Spirit Air Lines,10,10134,8.26,6.81,2066,20.4,1


# Q2 - Which airports have the highest cancellation rates? What are the primary reasons?

In [9]:
query2 = spark.sql("""
WITH airport_stats AS (
  SELECT
    ap.AIRPORT,
    COUNT(*) AS total_flights,
    SUM(f.CANCELLED) AS total_cancelled,
    ROUND(AVG(f.CANCELLED) * 100, 2) AS cancel_rate_pct,
    SUM(CASE WHEN f.CANCELLATION_REASON = 'A' THEN 1 ELSE 0 END) AS cancel_carrier,
    SUM(CASE WHEN f.CANCELLATION_REASON = 'B' THEN 1 ELSE 0 END) AS cancel_weather,
    SUM(CASE WHEN f.CANCELLATION_REASON = 'C' THEN 1 ELSE 0 END) AS cancel_nas,
    SUM(CASE WHEN f.CANCELLATION_REASON = 'D' THEN 1 ELSE 0 END) AS cancel_security
  FROM flights f
  JOIN airports ap ON f.ORIGIN_AIRPORT = ap.IATA_CODE
  GROUP BY ap.AIRPORT
  HAVING COUNT(*) >= 500
     AND SUM(f.CANCELLED) > 0
)
SELECT
  AIRPORT,
  cancel_rate_pct,
  total_flights,
  total_cancelled,
  cancel_carrier,
  cancel_weather,
  cancel_nas,
  cancel_security,
  CASE
    WHEN cancel_carrier >= cancel_weather AND cancel_carrier >= cancel_nas AND cancel_carrier >= cancel_security THEN 'Carrier-driven'
    WHEN cancel_weather >= cancel_carrier AND cancel_weather >= cancel_nas AND cancel_weather >= cancel_security THEN 'Weather-driven'
    WHEN cancel_nas >= cancel_carrier AND cancel_nas >= cancel_weather AND cancel_nas >= cancel_security THEN 'NAS-driven'
    ELSE 'Security-driven'
  END AS primary_cause
FROM airport_stats
ORDER BY cancel_rate_pct DESC
LIMIT 10;
""")
df2 = query2.toPandas()
print("\n Q2 - TOP 10 airports have the highest cancellation rates? What are the primary reasons? ")


 Q2 - TOP 10 airports have the highest cancellation rates? What are the primary reasons? 


,AIRPORT,cancel_rate_pct,total_flights,total_cancelled,cancel_carrier,cancel_weather,cancel_nas,cancel_security,primary_cause
0,Friedman Memorial Airport,9.21,956,88,2,84,2,0,Weather-driven
1,Devils Lake Regional Airport,8.76,525,46,7,14,25,0,NAS-driven
2,Aspen-Pitkin County Airport,7.75,3562,276,40,225,11,0,Weather-driven
3,Muskegon County Airport,7.35,667,49,6,39,4,0,Weather-driven
4,Jamestown Regional Airport,7.27,812,59,7,28,24,0,Weather-driven
5,Lawton-Fort Sill Regional Airport,7.06,1331,94,44,49,1,0,Weather-driven
6,Toledo Express Airport,6.76,962,65,13,31,21,0,Weather-driven
7,Houghton County Memorial Airport,6.74,668,45,5,38,2,0,Weather-driven
8,Barkley Regional Airport,6.61,666,44,7,34,3,0,Weather-driven
9,Jack Brooks Regional Airport (Southeast Texas ...,6.47,973,63,38,24,1,0,Carrier-driven


In [14]:
df2.head(10)

,AIRPORT,cancel_rate_pct,total_flights,total_cancelled,cancel_carrier,cancel_weather,cancel_nas,cancel_security,primary_cause
0,Friedman Memorial Airport,9.21,956,88,2,84,2,0,Weather-driven
1,Devils Lake Regional Airport,8.76,525,46,7,14,25,0,NAS-driven
2,Aspen-Pitkin County Airport,7.75,3562,276,40,225,11,0,Weather-driven
3,Muskegon County Airport,7.35,667,49,6,39,4,0,Weather-driven
4,Jamestown Regional Airport,7.27,812,59,7,28,24,0,Weather-driven
5,Lawton-Fort Sill Regional Airport,7.06,1331,94,44,49,1,0,Weather-driven
6,Toledo Express Airport,6.76,962,65,13,31,21,0,Weather-driven
7,Houghton County Memorial Airport,6.74,668,45,5,38,2,0,Weather-driven
8,Barkley Regional Airport,6.61,666,44,7,34,3,0,Weather-driven
9,Jack Brooks Regional Airport (Southeast Texas ...,6.47,973,63,38,24,1,0,Carrier-driven


# QUERY 3: The "Most Reliable" vs. "Worst" Routes for Each Airline


In [ ]:
query3 = spark.sql("""
    WITH RouteStats AS (
        SELECT
            al.AIRLINE as AirlineName,
            ap1.CITY as OriginCity,
            ap2.CITY as DestCity,
            COUNT(*) as total_flights,
            ROUND(AVG(f.ARRIVAL_DELAY), 2) as avg_arrival_delay,
            RANK() OVER(PARTITION BY al.AIRLINE ORDER BY AVG(f.ARRIVAL_DELAY) ASC) as reliable_rank,
            RANK() OVER(PARTITION BY al.AIRLINE ORDER BY AVG(f.ARRIVAL_DELAY) DESC) as worst_rank
        FROM flights f
        JOIN airlines al ON f.AIRLINE = al.IATA_CODE
        JOIN airports ap1 ON f.ORIGIN_AIRPORT = ap1.IATA_CODE
        JOIN airports ap2 ON f.DESTINATION_AIRPORT = ap2.IATA_CODE
        WHERE f.CANCELLED = 0 AND f.ARRIVAL_DELAY IS NOT NULL
        GROUP BY al.AIRLINE, ap1.CITY, ap2.CITY
        HAVING COUNT(*) > 100
    )
    SELECT
        AirlineName,
        OriginCity,
        DestCity,
        total_flights,
        avg_arrival_delay,
        CASE
            WHEN reliable_rank = 1 THEN 'Top 1 Reliable'
            WHEN worst_rank = 1 THEN 'Top 1 Worst'
        END as RouteCategory
    FROM RouteStats
    WHERE reliable_rank = 1 OR worst_rank = 1
    ORDER BY AirlineName, RouteCategory
""")
df3 = query3.toPandas()
print('\n Q3 - The "Most Reliable" vs. "Worst" Routes for Each Airline ')

In [20]:
df3.head(10)

,AirlineName,OriginCity,DestCity,total_flights,avg_arrival_delay,RouteCategory
0,Alaska Airlines Inc.,Kailua/Kona,San Diego,116,-22.47,Top 1 Reliable
1,Alaska Airlines Inc.,Kotzebue,Nome,265,11.50,Top 1 Worst
2,American Airlines Inc.,Kailua/Kona,Phoenix,105,-14.08,Top 1 Reliable
3,American Airlines Inc.,New York,Eagle,105,38.01,Top 1 Worst
4,American Eagle Airlines Inc.,Toledo,Chicago,897,-2.64,Top 1 Reliable
5,American Eagle Airlines Inc.,Aspen,Dallas-Fort Worth,262,39.97,Top 1 Worst
6,Atlantic Southeast Airlines,Boston,Newark,139,-12.72,Top 1 Reliable
7,Atlantic Southeast Airlines,Omaha,Minneapolis,116,30.35,Top 1 Worst
8,Delta Air Lines Inc.,Honolulu,Salt Lake City,323,-14.01,Top 1 Reliable
9,Delta Air Lines Inc.,San Francisco,Los Angeles,1352,27.64,Top 1 Worst


# Q4 - Which season has the most delays? (Quarterly + Seasonal Analysis)

In [10]:
query4 = spark.sql("""
WITH period AS (
  SELECT
    f.ARRIVAL_DELAY,
    f.DEPARTURE_DELAY,
    f.CANCELLED,
    f.WEATHER_DELAY,
    CASE
      WHEN f.MONTH IN (1,2,3)   THEN 'Q1 (Jan–Mar)'
      WHEN f.MONTH IN (4,5,6)   THEN 'Q2 (Apr–Jun)'
      WHEN f.MONTH IN (7,8,9)   THEN 'Q3 (Jul–Sep)'
      ELSE 'Q4 (Oct–Dec)'
    END AS quarter, CASE
      WHEN f.MONTH IN (12,1,2)  THEN 'Winter'
      WHEN f.MONTH IN (3,4,5)   THEN 'Spring'
      WHEN f.MONTH IN (6,7,8)   THEN 'Summer'
      ELSE 'Fall'
    END AS season
  FROM flights f
  WHERE f.ARRIVAL_DELAY IS NOT NULL
)
SELECT
  quarter,
  season,
  COUNT(*) AS total_flights,
  ROUND(AVG(ARRIVAL_DELAY), 1) AS avg_arr_delay,
  ROUND(AVG(DEPARTURE_DELAY), 1) AS avg_dep_delay,
  ROUND(AVG(CANCELLED) * 100, 2) AS cancel_rate_pct,
  ROUND(AVG(WEATHER_DELAY), 2) AS avg_weather_delay,
  ROUND(
    SUM(CASE WHEN ARRIVAL_DELAY > 15 THEN 1 ELSE 0 END)
    * 100 / COUNT(*), 1) AS pct_significantly_delayed
FROM period
GROUP BY quarter, season
ORDER BY quarter;
""")
df4 = query4.toPandas()
print("\n Q4 - Which season has the most delays? (Quarterly + Seasonal Analysis) ")


 Q4 - Which season has the most delays? (Quarterly + Seasonal Analysis) 


,quarter,season,total_flights,avg_arr_delay,avg_dep_delay,cancel_rate_pct,avg_weather_delay,pct_significantly_delayed
0,Q1 (Jan–Mar),Winter,864676,7.0,10.7,0.0,3.53,21.3
1,Q1 (Jan–Mar),Spring,492138,4.9,9.6,0.0,2.40,18.6
2,Q2 (Apr–Jun),Spring,968892,3.8,8.5,0.0,3.25,17.0
3,Q2 (Apr–Jun),Summer,492847,9.6,13.9,0.0,3.29,22.7
4,Q3 (Jul–Sep),Summer,1018340,5.5,10.6,0.0,2.47,19.1


In [17]:
df4.head()

,quarter,season,total_flights,avg_arr_delay,avg_dep_delay,cancel_rate_pct,avg_weather_delay,pct_significantly_delayed
0,Q1 (Jan–Mar),Winter,864676,7.0,10.7,0.0,3.53,21.3
1,Q1 (Jan–Mar),Spring,492138,4.9,9.6,0.0,2.40,18.6
2,Q2 (Apr–Jun),Spring,968892,3.8,8.5,0.0,3.25,17.0
3,Q2 (Apr–Jun),Summer,492847,9.6,13.9,0.0,3.29,22.7
4,Q3 (Jul–Sep),Summer,1018340,5.5,10.6,0.0,2.47,19.1


# Query 5 - Busiest Route and Top Performing Airlines


In [4]:
query5 = spark.sql("""
WITH top_route AS (
  SELECT
    ORIGIN_AIRPORT,
    DESTINATION_AIRPORT,
    COUNT(*) AS total_route_flights
  FROM flights
  GROUP BY ORIGIN_AIRPORT, DESTINATION_AIRPORT
  ORDER BY total_route_flights DESC
  LIMIT 1
),

airline_stats_on_route AS (
  SELECT
    tr.ORIGIN_AIRPORT,
    tr.DESTINATION_AIRPORT,
    al.AIRLINE AS airline_name,
    COUNT(*) AS flights_count,
    ROUND(AVG(f.CANCELLED) * 100, 2) AS cancel_rate_pct,
    ROUND(AVG(f.ARRIVAL_DELAY), 2) AS avg_arr_delay
  FROM flights f
  JOIN top_route tr
    ON f.ORIGIN_AIRPORT = tr.ORIGIN_AIRPORT
   AND f.DESTINATION_AIRPORT = tr.DESTINATION_AIRPORT
  LEFT JOIN airlines al
    ON f.AIRLINE = al.IATA_CODE
  GROUP BY tr.ORIGIN_AIRPORT, tr.DESTINATION_AIRPORT, al.AIRLINE
),

ranked_airlines AS (
  SELECT
    ORIGIN_AIRPORT,
    DESTINATION_AIRPORT,
    airline_name,
    flights_count,
    cancel_rate_pct,
    avg_arr_delay,
    -- Volumne
    DENSE_RANK() OVER (ORDER BY flights_count DESC) AS vol_rank,
    -- Cancel
    DENSE_RANK() OVER (ORDER BY cancel_rate_pct ASC) AS cancel_rank,
    --Delay
    DENSE_RANK() OVER (ORDER BY avg_arr_delay ASC) AS delay_rank
  FROM airline_stats_on_route
),

airline_ranking AS (
  SELECT
    ORIGIN_AIRPORT,
    DESTINATION_AIRPORT,
    airline_name,
    flights_count,
    cancel_rate_pct,
    avg_arr_delay,
    DENSE_RANK() OVER (ORDER BY (vol_rank + cancel_rank + delay_rank) ASC) AS performance_rank
  FROM ranked_airlines
)

SELECT
  ap_origin.AIRPORT AS origin_airport_name,
  ap_dest.AIRPORT AS destination_airport_name,
  ar.airline_name,
  ar.flights_count,
  ar.cancel_rate_pct,
  ar.avg_arr_delay,
  ar.performance_rank AS airline_performance_rank
FROM airline_ranking ar
JOIN airports ap_origin ON ar.ORIGIN_AIRPORT = ap_origin.IATA_CODE
JOIN airports ap_dest ON ar.DESTINATION_AIRPORT = ap_dest.IATA_CODE
ORDER BY airline_performance_rank ASC;
""")
df5 = query5.toPandas()
print("\n Q5 - Busiest Route and Top Performing Airlines ")


 Q5 - Busiest Route and Top Performing Airlines 


In [9]:
df5

,origin_airport_name,destination_airport_name,airline_name,flights_count,cancel_rate_pct,avg_arr_delay,airline_performance_rank
0,San Francisco International Airport,Los Angeles International Airport,American Airlines Inc.,1949,0.72,7.62,1
1,San Francisco International Airport,Los Angeles International Airport,United Air Lines Inc.,3624,0.83,9.07,1
2,San Francisco International Airport,Los Angeles International Airport,Virgin America,3069,0.59,9.32,1
3,San Francisco International Airport,Los Angeles International Airport,Southwest Airlines Co.,3084,8.24,10.59,2
4,San Francisco International Airport,Los Angeles International Airport,Delta Air Lines Inc.,1364,0.81,27.64,3
5,San Francisco International Airport,Los Angeles International Airport,Skywest Airlines Inc.,654,1.68,15.93,4
